In [1]:
import pandas as pd
import numpy as np
import os
import glob

data_path = "./data/"

csv_files = glob.glob(data_path + "*.csv")

dataframe = {}

for file in csv_files:
    name = os.path.basename(file).replace(".csv","")
    dataframe[name] = pd.read_csv(file)

In [2]:
#理解RFM模型
#R：最近消费时间
#F：消费频率
#M：消费金额

In [3]:
#需要三个表：订单信息；支付信息：客户信息

In [4]:
orders_rfm = dataframe["olist_orders_dataset"].copy()
payments_rfm = dataframe["olist_order_payments_dataset"].copy()
customers_rfm = dataframe["olist_customers_dataset"].copy()

print("订单表",orders_rfm.shape)
print("支付表",payments_rfm.shape)
print("客户表",customers_rfm.shape)

订单表 (99441, 8)
支付表 (103886, 5)
客户表 (99441, 5)


In [5]:
#在复购分析中区分过了customer_id和customer_unique_id(订单对应的客户和真实用户唯一id)
print(orders_rfm.columns.tolist())
print(payments_rfm.columns.tolist())
print(customers_rfm.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [6]:
#确定有效订单
#需要的是完整的订单，所以分析delivered订单
rfm_orders = orders_rfm[orders_rfm["order_status"] == "delivered"].copy()

In [7]:
print('有效订单数',len(rfm_orders))
print("订单数量",rfm_orders["order_id"].nunique())

有效订单数 96478
订单数量 96478


In [8]:
#没有重复订单

In [9]:
#合并订单表和客户表
rfm_orders = rfm_orders.merge(
    customers_rfm[["customer_id","customer_unique_id"]],
    on = "customer_id",
    how = "left",
)
print(rfm_orders.shape)
print("缺失真实用户id",rfm_orders["customer_unique_id"].isna().sum())

(96478, 9)
缺失真实用户id 0


In [10]:
#计算用户消费金额
payment_summary = payments_rfm.groupby(
    "order_id",
)["payment_value"].sum().reset_index()
print(payment_summary.head())
print("支付汇总行数",len(payment_summary))

                           order_id  payment_value
0  00010242fe8c5a6d1ba2dd792cb16214          72.19
1  00018f77f2f0320c557190d7a144bdd3         259.83
2  000229ec398224ef6ca0657da4fc703e         216.87
3  00024acbcdf0a6daa1e931b038114c75          25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04
支付汇总行数 99440


In [11]:
#原始支付行有103886，同订单支付记录合并计算

In [12]:
rfm_orders = rfm_orders.merge(
    payment_summary,#合并表
    on = "order_id",
    how = "left",
)
print("合并后行数",len(rfm_orders))
print("支付金额缺失数量",rfm_orders["payment_value"].isna().sum())
print("合并后订单数量",rfm_orders["order_id"].nunique())#统计不重复值

合并后行数 96478
支付金额缺失数量 1
合并后订单数量 96478


In [13]:
#存在缺失支付金额
rfm_orders[rfm_orders["payment_value"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,payment_value
29811,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00,830d5b7aaa3b6f1e9ad63703bec97d23,NaN


In [14]:
#status为delivered，所以可能是原表的问题
#检查原表,id为：bfbd0f9bdef84302105ad712db648a6c,
print(payments_rfm[payments_rfm["order_id"] == "bfbd0f9bdef84302105ad712db648a6c"])

Empty DataFrame
Columns: [order_id, payment_sequential, payment_type, payment_installments, payment_value]
Index: []


In [15]:
#原支付表中不存在这个支付记录，就是一个已送达但是没有支付记录的订单

In [16]:
rfm_orders[
    rfm_orders["payment_value"].isna()
][["order_id", "customer_unique_id"]]

,order_id,customer_unique_id
29811,bfbd0f9bdef84302105ad712db648a6c,830d5b7aaa3b6f1e9ad63703bec97d23


In [17]:
#检查时间类型
print(rfm_orders["order_purchase_timestamp"].dtype)
#转换时间
rfm_orders["order_purchase_timestamp"] = pd.to_datetime(
    rfm_orders["order_purchase_timestamp"]
)

str


In [18]:
print(rfm_orders["order_purchase_timestamp"].max())

2018-08-29 15:00:37


In [19]:
#创建观察基准日
snapshot_date = rfm_orders["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
print("观察基准日",snapshot_date)

观察基准日 2018-08-30 00:00:00


In [20]:
#计算每位客户最近一次购买时间
rfm = rfm_orders.groupby("customer_unique_id").agg(
    last_purchase = ("order_purchase_timestamp", "max")
).reset_index()

print(rfm.head())
print("客户数量",len(rfm))

                 customer_unique_id       last_purchase
0  0000366f3b9a7992bf8c76cfdf3221e2 2018-05-10 10:56:27
1  0000b849f77a49e4a4ce2b2a4ca5be3f 2018-05-07 11:11:27
2  0000f46a3911fa3c0805444483337064 2017-03-10 21:05:03
3  0000f6ccb0745a6a4b88665a16c9f078 2017-10-12 20:29:41
4  0004aac84e0df4da2b147fca70cf8255 2017-11-14 19:45:42
客户数量 93358


In [21]:
#计算R值
rfm["recency"] = (
    snapshot_date - rfm["last_purchase"]
).dt.days

print(rfm[["customer_unique_id","last_purchase","recency"]].head())


                 customer_unique_id       last_purchase  recency
0  0000366f3b9a7992bf8c76cfdf3221e2 2018-05-10 10:56:27      111
1  0000b849f77a49e4a4ce2b2a4ca5be3f 2018-05-07 11:11:27      114
2  0000f46a3911fa3c0805444483337064 2017-03-10 21:05:03      537
3  0000f6ccb0745a6a4b88665a16c9f078 2017-10-12 20:29:41      321
4  0004aac84e0df4da2b147fca70cf8255 2017-11-14 19:45:42      288


In [22]:
#R值完整性
print(rfm["recency"].describe())

count    93358.000000
mean       237.478888
std        152.595050
min          0.000000
25%        114.000000
50%        218.000000
75%        346.000000
max        713.000000
Name: recency, dtype: float64


In [23]:
#计算F，客户的消费频次，按客户的ID进行分组
frequency = rfm_orders.groupby(
    "customer_unique_id"
)["order_id"].nunique().reset_index()#统计客户不重复订单数量,避免重复订单影响频次

frequency.columns = ["customer_unique_id","frequency"]#重命名列[1,2]

print(frequency.head())
print("客户数量",len(frequency))

                 customer_unique_id  frequency
0  0000366f3b9a7992bf8c76cfdf3221e2          1
1  0000b849f77a49e4a4ce2b2a4ca5be3f          1
2  0000f46a3911fa3c0805444483337064          1
3  0000f6ccb0745a6a4b88665a16c9f078          1
4  0004aac84e0df4da2b147fca70cf8255          1
客户数量 93358


In [24]:
#R和F的客户数量订单
print(len(rfm))
print(len(frequency))

93358
93358


In [25]:
#统计客户消费频次分布
print(frequency["frequency"].value_counts().sort_index())#根据索引统计不同值，按索引小到大排列

frequency
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64


In [26]:
#购买频次为1的占大多数；定义复购为一次以上
repeat_rate = (frequency["frequency"]>1).mean()#frequency["frequency"]是df，判断条件得出的结果是true和false，而.mean将1，0相加算平均，得1的占比

print("复购订单数",(frequency["frequency"]>1).sum())
print("复购率",repeat_rate)

复购订单数 2801
复购率 0.03000278497825575


In [27]:
#计算M,先合并RF
rfm = rfm.merge(
    frequency,
    on = "customer_unique_id",
    how = "left"
)
print(rfm.head())

                 customer_unique_id       last_purchase  recency  frequency
0  0000366f3b9a7992bf8c76cfdf3221e2 2018-05-10 10:56:27      111          1
1  0000b849f77a49e4a4ce2b2a4ca5be3f 2018-05-07 11:11:27      114          1
2  0000f46a3911fa3c0805444483337064 2017-03-10 21:05:03      537          1
3  0000f6ccb0745a6a4b88665a16c9f078 2017-10-12 20:29:41      321          1
4  0004aac84e0df4da2b147fca70cf8255 2017-11-14 19:45:42      288          1


In [28]:
#计算M，用户累计支付金额
#按真实客户ID汇总支付金额
print(rfm_orders["payment_value"].isna().sum())
monetary = rfm_orders.groupby(
    "customer_unique_id"
)["payment_value"].sum(min_count=1).reset_index()

monetary.columns = ["customer_unique_id","monetary"]

print(monetary.head())
print("客户数量",len(monetary))

1
                 customer_unique_id  monetary
0  0000366f3b9a7992bf8c76cfdf3221e2    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f     27.19
2  0000f46a3911fa3c0805444483337064     86.22
3  0000f6ccb0745a6a4b88665a16c9f078     43.62
4  0004aac84e0df4da2b147fca70cf8255    196.89
客户数量 93358


In [29]:
#合并RFM表
rfm = rfm.merge(
    monetary,
    on = "customer_unique_id",
    how = "left"
)
print(rfm.head())
print("客户数量",len(rfm))
print("金额缺失值",rfm["monetary"].isna().sum())

                 customer_unique_id       last_purchase  recency  frequency  \
0  0000366f3b9a7992bf8c76cfdf3221e2 2018-05-10 10:56:27      111          1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f 2018-05-07 11:11:27      114          1   
2  0000f46a3911fa3c0805444483337064 2017-03-10 21:05:03      537          1   
3  0000f6ccb0745a6a4b88665a16c9f078 2017-10-12 20:29:41      321          1   
4  0004aac84e0df4da2b147fca70cf8255 2017-11-14 19:45:42      288          1   

   monetary  
0    141.90  
1     27.19  
2     86.22  
3     43.62  
4    196.89  
客户数量 93358
金额缺失值 1


In [33]:
print(rfm[["recency", "frequency", "monetary"]].describe())

            recency     frequency      monetary
count  93358.000000  93358.000000  93357.000000
mean     237.478888      1.033420    165.198772
std      152.595050      0.209097    226.314579
min        0.000000      1.000000      9.590000
25%      114.000000      1.000000     63.060000
50%      218.000000      1.000000    107.780000
75%      346.000000      1.000000    182.560000
max      713.000000     15.000000  13664.080000


In [34]:
#r:平均距离最后一天购买在237天，中位数在218，说明一半客户在最后一天购买距离观察日不超过218天
#f：平均购买频次在1.03次，中位数是1，最大值是max，说明客户复购行为比较少，绝大多数都只有一次购买记录
#m：平均消费为165.20，中位数在107.78，最大消费为13664.08，平均值高于中位数，说明少数高消费客户拉高了平均值，count少1，说明对应缺失值

In [40]:
#f的分布非常集中，在15%也是1
#查看f分布
print(rfm["frequency"].count())
rfm["frequency"].value_counts().sort_index()

93358


frequency
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64

In [46]:
#复购占比
print(rfm[rfm["frequency"]>1]
      ["frequency"].count()
      / rfm["frequency"].count() *100 , "%")

3.0002784978255748 %


In [47]:
#由于大部分用户的f都是1，所以评分F=1为低频用户；F=2为有一定的复购行为；F>=3为高频用户
#查看R
rfm["recency"].describe()

count    93358.000000
mean       237.478888
std        152.595050
min          0.000000
25%        114.000000
50%        218.000000
75%        346.000000
max        713.000000
Name: recency, dtype: float64

In [48]:
#结合RF，1.客户最近一次购买时间差异较大；2.约97%客户只有一笔已送达订单；3.复购用户占比约3%
#查看M
rfm["monetary"].describe()

count    93357.000000
mean       165.198772
std        226.314579
min          9.590000
25%         63.060000
50%        107.780000
75%        182.560000
max      13664.080000
Name: monetary, dtype: float64

In [49]:
#1.平均值明显高于中位数，说明消费金额也存在偏右分布；2.最高消费金额远高于普通用户3.少数高消费客户可能对整体的销售额贡献较大
#评分
#R
rfm["R_score"] = pd.qcut(
    rfm["recency"],
    q = 5,
    labels = [5,4,3,2,1]
)

In [51]:
rfm["R_score"].value_counts().sort_index()

R_score
5    18802
4    18757
3    18723
2    18471
1    18605
Name: count, dtype: int64

In [56]:
#由于F大多在1集中
rfm["F_score"] = pd.cut(
    rfm["frequency"],
    bins=[0,1,2,float("inf")],
    labels=[1,2,3]
)
rfm["F_score"].value_counts().sort_index()

F_score
1    90557
2     2573
3      228
Name: count, dtype: int64

In [57]:
#后续客户运营可以重点关注首次购买用户的二次转换

In [58]:
#M评分
rfm["M_score"] = pd.qcut(
    rfm["monetary"],
    q = 5,
    labels = [1,2,3,4,5]
)
rfm["M_score"].value_counts().sort_index()

M_score
1    18673
2    18683
3    18658
4    18675
5    18668
Name: count, dtype: int64

In [59]:
#计算RFm综合评分
#由于存在一个monetary为NAN的数据
score_cols = rfm[
    ["R_score","F_score","M_score"]
].astype(float)

rfm["RFM_score"] = score_cols.sum(
    axis = 1,
    min_count = 3
)

In [62]:
print(rfm[["R_score","F_score","M_score","RFM_score"]].head())
print("缺失数量",rfm["RFM_score"].isna().sum())

  R_score F_score M_score  RFM_score
0       4       1       4        9.0
1       4       1       1        6.0
2       1       1       2        4.0
3       2       1       1        4.0
4       2       1       4        7.0
缺失数量 1


In [63]:
#客户价值分层
rfm["RFM_score"].describe()

count    93357.000000
mean         7.039558
std          2.052455
min          3.000000
25%          6.000000
50%          7.000000
75%          9.000000
max         13.000000
Name: RFM_score, dtype: float64

In [65]:
rfm["RFM_score"].value_counts().sort_index()

RFM_score
3.0      3873
4.0      7441
5.0     11505
6.0     14180
7.0     18082
8.0     14858
9.0     11252
10.0     7803
11.0     3939
12.0      373
13.0       51
Name: count, dtype: int64

In [66]:
#创建客户分层标签
rfm["customer_segment"] = pd.cut(
    rfm["RFM_score"],
    bins = [0,4,6,8,10,15],
    labels = [
        "低价值客户",#需要关注流失风险
        "较低价值客户",#需要促进复购
        "中等价值客户",#有提升空间
        "较高价值客户",#有一定价值
        "高价值客户"#综合表现较好
    ]
)
print(rfm["customer_segment"].value_counts().sort_index())

customer_segment
低价值客户     11314
较低价值客户    25685
中等价值客户    32940
较高价值客户    19055
高价值客户      4363
Name: count, dtype: int64


In [67]:
#验证是否合理
#R、F、M平均值
segment_analysis = rfm.groupby(
    "customer_segment",
    observed = False
).agg(
    customer_count = ("customer_segment", "count"),
    avg_recency = ("recency", "mean"),
    avg_frequency = ("frequency", "mean"),
    avg_monetary = ("monetary", "mean")
).reset_index()

print(segment_analysis)

  customer_segment  customer_count  avg_recency  avg_frequency  avg_monetary
0            低价值客户           11314   423.959077       1.000619     50.601701
1           较低价值客户           25685   305.254740       1.005490     84.174070
2           中等价值客户           32940   212.842957       1.021038    178.573422
3           较高价值客户           19055   119.779375       1.060771    256.288390
4            高价值客户            4363    54.837726       1.256933    440.559092


In [72]:
#计算每个客户层级有多少客户，哪一层贡献了最多的金额
segment_revenue = rfm.groupby(
    "customer_segment",
    observed = False
).agg(
    customer_count = ("customer_unique_id", "count"),
    total_monetary =("monetary", "sum")
).reset_index()

segment_revenue["reven_ratio"] =(
    segment_revenue["total_monetary"] / segment_revenue["total_monetary"].sum()#每个客户的消费金额/所有客户的消费金额
)
print(segment_revenue)

  customer_segment  customer_count  total_monetary  reven_ratio
0            低价值客户           11314       572507.65     0.037122
1           较低价值客户           25685      2162011.00     0.140186
2           中等价值客户           32940      5882208.53     0.381405
3           较高价值客户           19055      4883575.27     0.316653
4            高价值客户            4363      1922159.32     0.124634


In [77]:
segment_revenue["人数占比"] = (
    segment_revenue["customer_count"]/segment_revenue["customer_count"].sum()
)
print(segment_revenue)

  customer_segment  customer_count  total_monetary  reven_ratio      人数占比
0            低价值客户           11314       572507.65     0.037122  0.121191
1           较低价值客户           25685      2162011.00     0.140186  0.275127
2           中等价值客户           32940      5882208.53     0.381405  0.352839
3           较高价值客户           19055      4883575.27     0.316653  0.204109
4            高价值客户            4363      1922159.32     0.124634  0.046735


In [79]:
#1.高价值用户占客户群体约4.67%但贡献却是12.46%左右；说明这部分客户的单客价值较高，值得重点维护
#2.中等价值群体占比35.28%，消费贡献是38.14%，是主要的消费群体，是重点运用群体
#3.较高价值客户群体占比在20.41%，消费贡献在31.67%同样很高，是否可以通过某些方式提高复购率？
#4.有趣的是较高价值和高价值用户合计占比在25%左右，但是揽下了约44.13%的消费金额贡献，说明约四分之一的客户贡献了约一半的消费金额
rfm.to_csv("rfm.csv",index=False)#导出RFM的csv文件